In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 📊 Microgrid IDS - Data Exploration\n",
    "## Understanding Wireless Sensor Network Data for Cyber Attack Detection\n",
    "\n",
    "This notebook explores the WSN microgrid dataset, visualizes attack patterns, and analyzes feature distributions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import required libraries\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import plotly.express as px\n",
    "import plotly.graph_objects as go\n",
    "from plotly.subplots import make_subplots\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Set style\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette('husl')\n",
    "\n",
    "# Display settings\n",
    "pd.set_option('display.max_columns', None)\n",
    "pd.set_option('display.max_rows', 100)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the dataset\n",
    "df = pd.read_csv('../data/raw/wsn_microgrid_data.csv')\n",
    "print(f\"✅ Dataset loaded successfully!\")\n",
    "print(f\"📊 Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\")\n",
    "print(f\"\\n📋 Columns:\\n{', '.join(df.columns.tolist())}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Basic information\n",
    "print(\"📋 Dataset Info:\")\n",
    "print(\"-\" * 50)\n",
    "df.info()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Statistical summary\n",
    "print(\"📊 Statistical Summary:\")\n",
    "print(\"-\" * 50)\n",
    "df.describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "missing = df.isnull().sum()\n",
    "missing_pct = (missing / len(df)) * 100\n",
    "\n",
    "missing_df = pd.DataFrame({\n",
    "    'Missing Count': missing,\n",
    "    'Percentage': missing_pct\n",
    "})[missing > 0]\n",
    "\n",
    "if len(missing_df) > 0:\n",
    "    print(\"⚠️ Missing values found:\")\n",
    "    display(missing_df)\n",
    "else:\n",
    "    print(\"✅ No missing values found!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Attack type distribution\n",
    "attack_names = ['Normal', 'Blackhole', 'Grayhole', 'Flooding', \n",
    "                'Sybil', 'Sinkhole', 'TDMA Exploit', 'Hello Flood']\n",
    "\n",
    "attack_counts = df['attack_type'].value_counts().sort_index()\n",
    "\n",
    "# Create figure with subplots\n",
    "fig = make_subplots(rows=1, cols=2,\n",
    "                    subplot_titles=('Attack Type Distribution', 'Class Balance'),\n",
    "                    specs=[[{'type': 'bar'}, {'type': 'pie'}]])\n",
    "\n",
    "# Add bar chart\n",
    "fig.add_trace(\n",
    "    go.Bar(x=attack_names, y=attack_counts,\n",
    "           marker_color=['green', 'red', 'orange', 'yellow', 'purple', 'pink', 'blue', 'brown'],\n",
    "           text=attack_counts, textposition='auto'),\n",
    "    row=1, col=1\n",
    ")\n",
    "\n",
    "# Add pie chart\n",
    "fig.add_trace(\n",
    "    go.Pie(labels=attack_names, values=attack_counts,\n",
    "           hole=0.3, textinfo='percent+label'),\n",
    "    row=1, col=2\n",
    ")\n",
    "\n",
    "# Update layout\n",
    "fig.update_layout(height=500, showlegend=False,\n",
    "                  title_text=\"Attack Type Distribution Analysis\",\n",
    "                  title_x=0.5)\n",
    "fig.update_xaxes(title_text=\"Attack Type\", row=1, col=1)\n",
    "fig.update_yaxes(title_text=\"Count\", row=1, col=1)\n",
    "\n",
    "fig.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature distributions by attack type\n",
    "features = ['packet_delay_ms', 'hop_count', 'packet_loss_rate', \n",
    "            'energy_consumption_mwh', 'data_rate_kbps', 'battery_level']\n",
    "\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "axes = axes.ravel()\n",
    "\n",
    "colors = ['green', 'red', 'orange', 'yellow', 'purple', 'pink', 'blue', 'brown']\n",
    "\n",
    "for i, feature in enumerate(features):\n",
    "    for attack in range(8):\n",
    "        data = df[df['attack_type'] == attack][feature]\n",
    "        axes[i].hist(data, alpha=0.6, label=attack_names[attack], \n",
    "                    bins=30, color=colors[attack], edgecolor='black')\n",
    "    axes[i].set_title(f'{feature} Distribution by Attack Type', fontsize=12, fontweight='bold')\n",
    "    axes[i].set_xlabel(feature)\n",
    "    axes[i].set_ylabel('Frequency')\n",
    "    axes[i].legend(loc='upper right', fontsize=8)\n",
    "    axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Feature Distributions Across Different Attack Types', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Box plots for outlier detection\n",
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "axes = axes.ravel()\n",
    "\n",
    "for i, feature in enumerate(features):\n",
    "    data_to_plot = [df[df['attack_type'] == attack][feature] for attack in range(8)]\n",
    "    bp = axes[i].boxplot(data_to_plot, labels=attack_names, patch_artist=True)\n",
    "    \n",
    "    # Color boxes\n",
    "    for patch, color in zip(bp['boxes'], colors):\n",
    "        patch.set_facecolor(color)\n",
    "        patch.set_alpha(0.6)\n",
    "    \n",
    "    axes[i].set_title(f'{feature} Box Plot', fontsize=12, fontweight='bold')\n",
    "    axes[i].set_ylabel(feature)\n",
    "    axes[i].tick_params(axis='x', rotation=45)\n",
    "    axes[i].grid(True, alpha=0.3)\n",
    "\n",
    "plt.suptitle('Outlier Analysis - Box Plots by Attack Type', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation matrix\n",
    "numeric_cols = df.select_dtypes(include=[np.number]).columns\n",
    "corr_matrix = df[numeric_cols].corr()\n",
    "\n",
    "plt.figure(figsize=(14, 10))\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', \n",
    "            center=0, square=True, linewidths=1, cbar_kws={\"shrink\": 0.8})\n",
    "plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Pair plot for selected features (sample for visualization)\n",
    "selected_features = ['packet_delay_ms', 'packet_loss_rate', 'energy_consumption_mwh', 'data_rate_kbps']\n",
    "sample_df = df.sample(n=1000, random_state=42)  # Sample for performance\n",
    "\n",
    "g = sns.pairplot(data=sample_df, vars=selected_features, \n",
    "                 hue='attack_type', palette=colors, \n",
    "                 diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30})\n",
    "g.fig.suptitle('Pair Plot of Selected Features', y=1.02, fontsize=16, fontweight='bold')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Time series analysis (if timestamp exists)\n",
    "if 'timestamp' in df.columns:\n",
    "    df['timestamp'] = pd.to_datetime(df['timestamp'])\n",
    "    df.set_index('timestamp', inplace=True)\n",
    "    \n",
    "    # Resample by hour\n",
    "    hourly_attacks = df['attack_type'].resample('H').apply(lambda x: (x > 0).sum())\n",
    "    \n",
    "    plt.figure(figsize=(15, 6))\n",
    "    plt.plot(hourly_attacks.index, hourly_attacks.values, \n",
    "             color='red', linewidth=2, marker='o', markersize=4)\n",
    "    plt.fill_between(hourly_attacks.index, 0, hourly_attacks.values, \n",
    "                     alpha=0.3, color='red')\n",
    "    plt.title('Hourly Attack Frequency Over Time', fontsize=16, fontweight='bold')\n",
    "    plt.xlabel('Time')\n",
    "    plt.ylabel('Number of Attacks')\n",
    "    plt.grid(True, alpha=0.3)\n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Statistical tests\n",
    "from scipy import stats\n",
    "\n",
    "print(\"📊 Statistical Tests for Feature Significance:\")\n",
    "print(\"-\" * 50)\n",
    "\n",
    "for feature in features:\n",
    "    # ANOVA test\n",
    "    groups = [df[df['attack_type'] == i][feature] for i in range(8)]\n",
    "    f_stat, p_value = stats.f_oneway(*groups)\n",
    "    \n",
    "    significance = \"✅ Significant\" if p_value < 0.05 else \"❌ Not Significant\"\n",
    "    print(f\"{feature:25} | F-statistic: {f_stat:8.4f} | p-value: {p_value:8.6f} | {significance}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Key insights\n",
    "print(\"\\n🔍 Key Insights:\")\n",
    "print(\"-\" * 50)\n",
    "\n",
    "print(\"1. Attack Distribution:\")\n",
    "for i, name in enumerate(attack_names):\n",
    "    count = attack_counts[i]\n",
    "    pct = (count / len(df)) * 100\n",
    "    print(f\"   • {name:15}: {count:6,} samples ({pct:.2f}%)\")\n",
    "\n",
    "print(\"\\n2. Most Discriminative Features (by ANOVA):\")\n",
    "f_scores = {}\n",
    "for feature in features:\n",
    "    groups = [df[df['attack_type'] == i][feature] for i in range(8)]\n",
    "    f_stat, _ = stats.f_oneway(*groups)\n",
    "    f_scores[feature] = f_stat\n",
    "\n",
    "sorted_features = sorted(f_scores.items(), key=lambda x: x[1], reverse=True)\n",
    "for i, (feat, score) in enumerate(sorted_features[:3]):\n",
    "    print(f\"   {i+1}. {feat}: F-statistic = {score:.2f}\")\n",
    "\n",
    "print(\"\\n3. Data Quality:\")\n",
    "print(f\"   • Total samples: {len(df):,}\")\n",
    "print(f\"   • Unique nodes: {df['node_id'].nunique()}\")\n",
    "print(f\"   • Features: {len(df.columns)}\")\n",
    "print(f\"   • Missing values: {df.isnull().sum().sum()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save processed data for modeling\n",
    "print(\"\\n💾 Saving processed data...\")\n",
    "\n",
    "# Save feature distributions\n",
    "feature_stats = df[features + ['attack_type']].groupby('attack_type').describe()\n",
    "feature_stats.to_csv('../results/feature_statistics.csv')\n",
    "\n",
    "# Save correlation matrix\n",
    "corr_matrix.to_csv('../results/correlation_matrix.csv')\n",
    "\n",
    "print(\"✅ Data exploration complete! Results saved to 'results/' folder\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.9.0"
  }
 }
}